# 8. 누수 피처 최종 점검

`X_model.csv`를 기준으로 식별자, 타겟 변수, 타겟에서 직접 파생된 피처를 구분하고 모델 입력에서 제외할 피처를 점검한다.

타겟 변수는 다음 두 가지이다.

- `daily_views_log1p`
- `scrap_rate (%)`

In [9]:
import pandas as pd
import numpy as np

df = pd.read_csv("X_model.csv")

print("데이터 크기:", df.shape)
print("전체 열 수:", len(df.columns))

데이터 크기: (7264, 86)
전체 열 수: 86


In [10]:
id_cols = [
    "activity_id"
]

target_cols = [
    "daily_views_log1p",
    "scrap_rate (%)"
]

leakage_cols = [
    "views_log1p",
    "scrap_count_log1p",
    "daily_views",
    "daily_scrap_count",
    "daily_scrap_count_log1p"
]

def classify_feature(column):
    if column in id_cols:
        return "식별자"
    if column in target_cols:
        return "타겟"
    if column in leakage_cols:
        return "타겟 직접 파생"
    return "사용 가능"

audit_table = pd.DataFrame({
    "feature": df.columns,
    "classification": [
        classify_feature(column) for column in df.columns
    ]
})

print("=== 피처 분류 결과 ===")
print(audit_table["classification"].value_counts())

print("\n=== 모델 입력에서 제외할 열 ===")
display(
    audit_table[
        audit_table["classification"] != "사용 가능"
    ]
)

=== 피처 분류 결과 ===
classification
사용 가능       78
타겟 직접 파생     5
타겟           2
식별자          1
Name: count, dtype: int64

=== 모델 입력에서 제외할 열 ===


,feature,classification
0,activity_id,식별자
2,scrap_rate (%),타겟
4,views_log1p,타겟 직접 파생
5,scrap_count_log1p,타겟 직접 파생
82,daily_views,타겟 직접 파생
83,daily_scrap_count,타겟 직접 파생
84,daily_views_log1p,타겟
85,daily_scrap_count_log1p,타겟 직접 파생


In [11]:
# 로그값에서 원래 조회수와 스크랩수 복원
views_original = np.rint(
    np.expm1(df["views_log1p"])
)

scrap_original = np.rint(
    np.expm1(df["scrap_count_log1p"])
)

# 스크랩 전환율 재계산
calculated_scrap_rate = np.where(
    views_original >= 10,
    np.round(scrap_original / views_original * 100, 4),
    0
)

# 일평균 조회수·스크랩수 재계산
calculated_daily_views = np.round(
    views_original / df["recruit_period_days"],
    4
)

calculated_daily_scrap = np.round(
    scrap_original / df["recruit_period_days"],
    4
)

# 일평균 값의 로그 변환 재계산
calculated_daily_views_log = np.round(
    np.log1p(calculated_daily_views),
    4
)

calculated_daily_scrap_log = np.round(
    np.log1p(calculated_daily_scrap),
    4
)

verification = pd.Series({
    "scrap_rate (%)": np.allclose(
        df["scrap_rate (%)"],
        calculated_scrap_rate,
        atol=1e-4,
        equal_nan=True
    ),
    "daily_views": np.allclose(
        df["daily_views"],
        calculated_daily_views,
        atol=1e-4,
        equal_nan=True
    ),
    "daily_scrap_count": np.allclose(
        df["daily_scrap_count"],
        calculated_daily_scrap,
        atol=1e-4,
        equal_nan=True
    ),
    "daily_views_log1p": np.allclose(
        df["daily_views_log1p"],
        calculated_daily_views_log,
        atol=1e-4,
        equal_nan=True
    ),
    "daily_scrap_count_log1p": np.allclose(
        df["daily_scrap_count_log1p"],
        calculated_daily_scrap_log,
        atol=1e-4,
        equal_nan=True
    )
})

print("=== 타겟 및 파생 피처 계산 관계 검증 ===")
display(verification.to_frame("계산 일치 여부"))

=== 타겟 및 파생 피처 계산 관계 검증 ===


,계산 일치 여부
scrap_rate (%),True
daily_views,True
daily_scrap_count,True
daily_views_log1p,True
daily_scrap_count_log1p,True


In [12]:
# 식별자는 별도로 보관
activity_ids = df[id_cols].copy()

# 두 타겟 분리
y = df[target_cols].copy()

# 식별자·타겟·타겟 직접 파생 피처 제외
X = df.drop(
    columns=id_cols + target_cols + leakage_cols
).copy()

# 결과 관련 이름이 X에 남아 있는지 검사
outcome_keywords = [
    "view",
    "scrap"
]

remaining_outcome_columns = [
    column
    for column in X.columns
    if any(keyword in column.lower() for keyword in outcome_keywords)
]

# 구조 검증
assert len(X) == len(y) == len(activity_ids)
assert set(id_cols).isdisjoint(X.columns)
assert set(target_cols).isdisjoint(X.columns)
assert set(leakage_cols).isdisjoint(X.columns)

print("=== 최종 분리 결과 ===")
print("원본 데이터:", df.shape)
print("모델 입력 X:", X.shape)
print("타겟 y:", y.shape)
print("식별자:", activity_ids.shape)
print("X에 남은 결과 관련 열:", remaining_outcome_columns)

=== 최종 분리 결과 ===
원본 데이터: (7264, 86)
모델 입력 X: (7264, 78)
타겟 y: (7264, 2)
식별자: (7264, 1)
X에 남은 결과 관련 열: []


In [13]:
missing_summary = (
    X.isna()
    .sum()
    .loc[lambda values: values > 0]
    .sort_values(ascending=False)
)

print("=== 모델 입력 X의 결측치 ===")

if missing_summary.empty:
    print("결측치 없음")
else:
    display(
        missing_summary.to_frame("결측치 수")
    )

=== 모델 입력 X의 결측치 ===


,결측치 수
activity_period_months,160


## 누수 피처 최종 점검 결과

### 1. 타겟 변수

본 프로젝트에서는 다음 두 변수를 타겟으로 사용한다.

- `daily_views_log1p`
- `scrap_rate (%)`

두 변수는 정답에 해당하므로 모델 입력 X에서는 제외하고 y로 분리한다.

### 2. 식별자 제외

`activity_id`는 공고를 구분하기 위한 식별자이며 공고의 특성을 나타내는 피처가 아니다. 모델이 식별자에 포함된 의미 없는 패턴을 학습하지 않도록 X에서 제외하고, 예측 결과를 원본 공고와 연결하기 위한 용도로만 별도 보관한다.

### 3. 타겟 직접 파생 피처 제외

다음 피처는 조회수 또는 스크랩수로부터 직접 계산된 결과 정보이므로 모델 입력에 포함하면 타겟 누수가 발생한다.

- `views_log1p`
- `scrap_count_log1p`
- `daily_views`
- `daily_scrap_count`
- `daily_scrap_count_log1p`

저장된 값과 직접 재계산한 값을 비교한 결과, 모든 파생 관계가 반올림 허용 오차 범위 내에서 일치했다.

### 4. 최종 분리 결과

- 원본 데이터: 7,264행 × 86열
- 모델 입력 X: 7,264행 × 78열
- 타겟 y: 7,264행 × 2열
- 식별자: 7,264행 × 1열
- X에 남아 있는 조회수·스크랩 관련 결과 피처: 없음

나머지 78개 피처는 공고 등록 시점에 확인할 수 있는 공고의 속성 및 파생 정보이므로 모델 입력으로 사용할 수 있다.

### 5. 전처리 누수 주의

모델 입력 X에서 `activity_period_months`에 160개의 결측치가 확인되었다.

전체 데이터를 이용해 중앙값을 계산한 후 결측치를 대치하면 검증·테스트 데이터의 정보가 학습 과정에 반영될 수 있다. 따라서 모델링 단계에서는 학습·검증 데이터를 먼저 분리하고, 학습 데이터에서만 중앙값을 계산한 뒤 동일한 값을 검증·테스트 데이터에 적용해야 한다.

### 최종 결론

식별자 1개, 타겟 2개, 타겟에서 직접 파생된 피처 5개를 모델 입력에서 제외했다. 최종 모델 입력 X에는 직접적인 타겟 누수 피처가 남아 있지 않다.